<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/SDXL-LORA-TRAINING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SDXL LoRA training (free Colab T4)

Trains an SDXL LoRA with [kohya-ss sd-scripts](https://github.com/kohya-ss/sd-scripts) and saves it to
`MyDrive/ComfyUI/loras_sdxl`, so it shows up in the website's LoRA list the next time you start the BigLust notebook.

**Before you start**
1. Collect **15–40 images** of the subject (jpg/png/webp, varied poses/backgrounds/lighting, clear subject, ideally
   1024 px or larger). Optional: a caption `.txt` per image with the same name (`photo1.jpg` → `photo1.txt`), e.g.
   `mytrigger, woman, red dress, standing in a park`. Images without one get the trigger word (and class word) as caption.
2. **Runtime → Change runtime type → T4 GPU.** Don't run the BigLust or Qwen notebook at the same time — training needs the whole GPU.

**Getting the images in:** cell 3 opens a file picker (pick the images, or one **.zip** with all of them — much faster).
They're saved in Google Drive under `MyDrive/ComfyUI/training/<your LoRA name>/images`, which cell 2 creates. You can also
upload into that folder with the Google Drive website/app instead; in Colab's file panel it's under
**content → drive → MyDrive → ComfyUI → training** (it only appears after cell 2 has run — press the refresh button).

**Run** cells 1 → 5. Training usually takes 1–3 hours on a T4 (the progress bar shows the time left). Progress is saved to Drive every
250 steps: if Colab disconnects, open the notebook again, use the same settings and run all cells — it continues where it stopped.

Only train on images you have the rights to use, and only train a LoRA of a real person with that person's consent.

In [ ]:
# Cell 1: settings
#@markdown **Name** of the LoRA. Your images go in `MyDrive/ComfyUI/training/<name>/images`.
LORA_NAME = "my_lora"  #@param {type:"string"}
#@markdown **Trigger word** — a made-up word you'll put in prompts to activate the LoRA (e.g. `zxq_anna`).
TRIGGER_WORD = "zxq_subject"  #@param {type:"string"}
#@markdown **Class word** — what the subject is (`woman`, `man`, `dog`, `style`, ...). Used for images without a caption file.
CLASS_WORD = "woman"  #@param {type:"string"}
#@markdown Add the trigger word to the start of existing captions that don't contain it.
ADD_TRIGGER_TO_CAPTIONS = True  #@param {type:"boolean"}

#@markdown ---
#@markdown **Base model** — train on the checkpoint you'll generate with. `biglust` = the BigLust notebook's model.
#@markdown Or paste a Civitai/Hugging Face download link, or a path like `/content/drive/MyDrive/models/x.safetensors`.
BASE_MODEL = "biglust"  #@param {type:"string"}

#@markdown ---
#@markdown **Training length.** ~1500 steps suits 15–40 images. More images → more steps.
MAX_TRAIN_STEPS = 1500  #@param {type:"integer"}
#@markdown How many times each image is seen per epoch (only changes how steps are grouped; 10 is fine).
NUM_REPEATS = 10  #@param {type:"integer"}
#@markdown Images per step. 2 fits a T4 at 1024 px; use 1 if you get "CUDA out of memory".
TRAIN_BATCH_SIZE = 2  #@param {type:"integer"}
RESOLUTION = 1024  #@param [768, 896, 1024] {type:"raw"}
#@markdown LoRA size: dim 8 (alpha 4) is a good default; 16/8 learns more detail but the file is bigger.
NETWORK_DIM = 8  #@param {type:"integer"}
NETWORK_ALPHA = 4  #@param {type:"integer"}
LEARNING_RATE = 3e-4  #@param {type:"number"}
SEED = 42  #@param {type:"integer"}
#@markdown Save a checkpoint (and resumable state) to Drive every N steps.
SAVE_EVERY_N_STEPS = 250  #@param {type:"integer"}

import re
LORA_NAME = re.sub(r"[^\w\-]+", "_", LORA_NAME.strip()).strip("_") or "my_lora"
DRIVE_ROOT = "/content/drive/MyDrive/ComfyUI"
TRAIN_DIR = f"{DRIVE_ROOT}/training/{LORA_NAME}"
IMAGES_DIR = f"{TRAIN_DIR}/images"
OUTPUT_DIR = f"{TRAIN_DIR}/output"        # step checkpoints + resumable state
LORAS_SDXL = f"{DRIVE_ROOT}/loras_sdxl"   # the final LoRA goes here (BigLust notebook reads it)
print(f"LoRA '{LORA_NAME}': images from {IMAGES_DIR}")
print(f"Final LoRA will be saved as {LORAS_SDXL}/{LORA_NAME}.safetensors")

In [ ]:
# Cell 2: connect Google Drive, install sd-scripts, download the base model
%cd /content
import os, subprocess, hashlib

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LORAS_SDXL, exist_ok=True)

# kohya-ss sd-scripts, pinned to a tested commit so updates can't break the notebook
SD_SCRIPTS_COMMIT = "cb70b96"
if not os.path.exists("/content/sd-scripts"):
    !git clone -q https://github.com/kohya-ss/sd-scripts /content/sd-scripts
!cd /content/sd-scripts && git fetch -q origin && git checkout -q {SD_SCRIPTS_COMMIT}
!cd /content/sd-scripts && pip install -q -r requirements.txt
!apt install -qqy aria2 > /dev/null

# Base model
os.makedirs("/content/base_models", exist_ok=True)
CIVITAI_TOKEN = "777a5632707c15ecf54e13eb3b1b862b"
BIGLUST_URL = f"https://civitai.com/api/download/models/1081768?fileId=986850&token={CIVITAI_TOKEN}"

base = BASE_MODEL.strip()
if base.lower() == "biglust":
    base = BIGLUST_URL
if base.startswith("http"):
    url = base.replace("/blob/", "/resolve/")
    if "civitai.com" in url and "token=" not in url:
        url += ("&" if "?" in url else "?") + "token=" + CIVITAI_TOKEN
    BASE_MODEL_PATH = f"/content/base_models/base_{hashlib.sha1(url.encode()).hexdigest()[:10]}.safetensors"
    if not os.path.exists(BASE_MODEL_PATH):
        print("Downloading base model...")
        !wget -q --show-progress -O {BASE_MODEL_PATH}.part "{url}" && mv {BASE_MODEL_PATH}.part {BASE_MODEL_PATH}
else:
    BASE_MODEL_PATH = base
if not os.path.exists(BASE_MODEL_PATH) or os.path.getsize(BASE_MODEL_PATH) < 1_000_000_000:
    raise RuntimeError(f"Base model missing or incomplete: {BASE_MODEL_PATH}. Check BASE_MODEL in cell 1.")

# SDXL VAE (same one the BigLust notebook uses)
VAE_PATH = "/content/base_models/sdxl_vae.safetensors"
if not os.path.exists(VAE_PATH):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors -d /content/base_models -o sdxl_vae.safetensors

print(f"Base model: {BASE_MODEL_PATH} ({os.path.getsize(BASE_MODEL_PATH) / 1e9:.1f} GB)")

In [ ]:
# Cell 3: upload training images (opens a file picker)
#@markdown Pick your images (and optional caption `.txt` files) — or one **.zip** with all of them, which is much
#@markdown faster for many images. They're saved to your Drive folder from cell 1.
#@markdown If the folder already has images this cell skips the picker (so "Run all" can resume training);
#@markdown tick this box to add more anyway.
ADD_MORE_IMAGES = False  #@param {type:"boolean"}

import os, io, zipfile
from google.colab import files

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
OK_EXTS = IMAGE_EXTS + (".txt",)


def save_upload(name, data):
    """Save one uploaded file (or the images/captions inside a zip) into IMAGES_DIR."""
    saved, skipped = 0, []
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            for member in z.namelist():
                base = os.path.basename(member)
                if member.endswith("/") or base.startswith(".") or "__MACOSX" in member:
                    continue
                if base.lower().endswith(OK_EXTS):
                    with open(os.path.join(IMAGES_DIR, base), "wb") as fh:
                        fh.write(z.read(member))
                    saved += 1
                else:
                    skipped.append(base)
    elif name.lower().endswith(OK_EXTS):
        with open(os.path.join(IMAGES_DIR, os.path.basename(name)), "wb") as fh:
            fh.write(data)
        saved += 1
    else:
        skipped.append(name)
    return saved, skipped


os.makedirs(IMAGES_DIR, exist_ok=True)
existing = [f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(IMAGE_EXTS)]
if existing and not ADD_MORE_IMAGES:
    print(f"{len(existing)} images already in {IMAGES_DIR} — skipping upload. "
          "Tick ADD_MORE_IMAGES above to upload more.")
else:
    print(f"Choose images, caption .txt files or a .zip. They'll be saved to {IMAGES_DIR}")
    uploaded = files.upload()
    total, all_skipped = 0, []
    for name, data in uploaded.items():
        saved, skipped = save_upload(name, data)
        total += saved
        all_skipped += skipped
        if os.path.exists(name):
            os.remove(name)  # files.upload also writes a copy to the working folder
    n = len([f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(IMAGE_EXTS)])
    print(f"Saved {total} file(s). The folder now has {n} images.")
    if all_skipped:
        print(f"Skipped (not jpg/png/webp/bmp/txt): {', '.join(all_skipped[:10])}"
              + (" ..." if len(all_skipped) > 10 else "")
              + "  — iPhone HEIC photos need converting to jpg first.")

In [ ]:
# Cell 4: prepare the dataset (copied to the runtime; your Drive files are not changed)
import os, shutil, math

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
LOCAL_DATA = f"/content/dataset/{LORA_NAME}"
shutil.rmtree(LOCAL_DATA, ignore_errors=True)
os.makedirs(LOCAL_DATA)

images = sorted(f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(IMAGE_EXTS))
if not images:
    raise RuntimeError(f"No images found in {IMAGES_DIR}. Upload 15-40 images with cell 3 (or into that Drive folder) and run this cell again.")

default_caption = ", ".join(w for w in (TRIGGER_WORD.strip(), CLASS_WORD.strip()) if w)
made, prefixed = 0, 0
for f in images:
    shutil.copy2(os.path.join(IMAGES_DIR, f), os.path.join(LOCAL_DATA, f))
    stem = os.path.splitext(f)[0]
    src_caption = os.path.join(IMAGES_DIR, stem + ".txt")
    caption = open(src_caption, encoding="utf-8").read().strip() if os.path.exists(src_caption) else ""
    if not caption:
        caption, made = default_caption, made + 1
    elif ADD_TRIGGER_TO_CAPTIONS and TRIGGER_WORD and TRIGGER_WORD.lower() not in caption.lower():
        caption, prefixed = f"{TRIGGER_WORD}, {caption}", prefixed + 1
    with open(os.path.join(LOCAL_DATA, stem + ".txt"), "w", encoding="utf-8") as fh:
        fh.write(caption)

DATASET_TOML = "/content/dataset/dataset.toml"
with open(DATASET_TOML, "w") as fh:
    fh.write(f'''[general]
enable_bucket = true
min_bucket_reso = 512
max_bucket_reso = {max(RESOLUTION * 3 // 2, 1024)}
bucket_reso_steps = 64

[[datasets]]
resolution = {RESOLUTION}
batch_size = {TRAIN_BATCH_SIZE}

  [[datasets.subsets]]
  image_dir = "{LOCAL_DATA}"
  num_repeats = {NUM_REPEATS}
  caption_extension = ".txt"
''')

per_epoch = math.ceil(len(images) * NUM_REPEATS / TRAIN_BATCH_SIZE)
print(f"{len(images)} images ({made} captioned automatically as '{default_caption}', {prefixed} got the trigger word added)")
print(f"{MAX_TRAIN_STEPS} steps ≈ {MAX_TRAIN_STEPS / per_epoch:.1f} epochs. "
      "The progress bar in cell 5 shows the real speed and time left.")
if len(images) < 10:
    print("Warning: fewer than 10 images usually gives a weak or overfitted LoRA.")

In [ ]:
# Cell 5: train (continues automatically from the last saved step if a previous run was interrupted)
%cd /content/sd-scripts
import os, glob, re, shutil

states = sorted(glob.glob(f"{OUTPUT_DIR}/{LORA_NAME}-step*-state"),
                key=lambda p: int(re.search(r"-step(\d+)-state", p).group(1)))
resume_args = ""
if states:
    print(f"Resuming from {os.path.basename(states[-1])}")
    resume_args = f'--resume "{states[-1]}"'

# Settings that fit a free T4 (15 GB VRAM, ~12.7 GB RAM):
# load straight to the GPU in fp16, train only the U-Net with cached text-encoder outputs and
# latents, gradient checkpointing, 8-bit AdamW, SDPA attention, VAE kept in fp32 (fp16 SDXL VAE gives NaNs).
cmd = f"""python sdxl_train_network.py
  --pretrained_model_name_or_path="{BASE_MODEL_PATH}"
  --vae="{VAE_PATH}"
  --dataset_config="{DATASET_TOML}"
  --output_dir="{OUTPUT_DIR}"
  --output_name="{LORA_NAME}"
  --save_model_as=safetensors
  --save_precision=fp16
  --network_module=networks.lora
  --network_dim={NETWORK_DIM}
  --network_alpha={NETWORK_ALPHA}
  --network_train_unet_only
  --cache_text_encoder_outputs
  --cache_latents
  --cache_latents_to_disk
  --learning_rate={LEARNING_RATE}
  --unet_lr={LEARNING_RATE}
  --lr_scheduler=cosine
  --lr_warmup_steps={max(1, MAX_TRAIN_STEPS // 20)}
  --optimizer_type=AdamW8bit
  --max_train_steps={MAX_TRAIN_STEPS}
  --mixed_precision=fp16
  --full_fp16
  --lowram
  --no_half_vae
  --gradient_checkpointing
  --sdpa
  --max_data_loader_n_workers=1
  --seed={SEED}
  --save_every_n_steps={SAVE_EVERY_N_STEPS}
  --save_state
  --save_last_n_steps_state={SAVE_EVERY_N_STEPS * 2}
  {resume_args}
""".replace("\n", " ")
!{cmd}

final = f"{OUTPUT_DIR}/{LORA_NAME}.safetensors"
if os.path.exists(final):
    dest = f"{LORAS_SDXL}/{LORA_NAME}.safetensors"
    shutil.copy2(final, dest)
    print("=" * 60)
    print(f"Done! LoRA saved to {dest}")
    print(f"Start the BigLust notebook, pick '{LORA_NAME}.safetensors' in the website's LoRA list,")
    print(f"and use '{TRIGGER_WORD}' in your prompt. Start at strength 0.7-1.0.")
    print(f"Earlier checkpoints (in case the final one is overcooked): {OUTPUT_DIR}")
    print("=" * 60)
else:
    print("Training did not finish. Check the error above; if Colab disconnected, run all cells again to resume.")